# ARC-AGI-2 - curricular symbolic solver submission (ADR 0104)

CPU only, no internet, no model. The solver source (`src/curriculum`) is embedded below as a
base64 zip and extracted at run time, so the notebook runs exactly the tested code. Solver:
verified candidates from the curricular library (main, objects, sequences), simplicity ranking,
up to two distinct attempts, ADR 0011 fallback when no candidate verifies. Each task runs in its
own process with a hard timeout; a global wall budget guards the total run time.

Output: `/kaggle/working/submission.json` (the only filename Kaggle accepts, ADR 0049).

In [ ]:
import base64, hashlib, io, os, sys, zipfile

CODE_SHA256 = "37df6a5d9bd821bb6655ff586828636346ff410860722c16c91702b27c0edf1b"
CODE_B64 = "..."

_raw = base64.b64decode(CODE_B64)
assert hashlib.sha256(_raw).hexdigest() == CODE_SHA256, "embedded solver zip is corrupted"
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
SRC_ROOT = os.path.join(WORK_DIR, "curricular_src")
zipfile.ZipFile(io.BytesIO(_raw)).extractall(SRC_ROOT)
sys.path.insert(0, SRC_ROOT)
print("solver extracted to", SRC_ROOT, "sha256", CODE_SHA256[:12])


In [ ]:
import os
from pathlib import Path

from src.curriculum.parallel_batch import default_worker_count
from src.curriculum.submission.build import build_submission_file


def find_challenges_path():
    for root, _, files in os.walk("/kaggle/input"):
        for name in files:
            if name.endswith("test_challenges.json"):
                return Path(root) / name
    raise FileNotFoundError("no *test_challenges.json under /kaggle/input")


CHALLENGES = find_challenges_path()
OUTPUT = Path(WORK_DIR) / "submission.json"
REPORT = Path(WORK_DIR) / "submission_report.json"
WORKERS = default_worker_count()
print("challenges:", CHALLENGES, "| cpus:", os.cpu_count(), "| workers:", WORKERS)
SUMMARY = build_submission_file(CHALLENGES, OUTPUT, REPORT, WORKERS)
print("wall seconds:", SUMMARY["wall_seconds"], "| statuses:", SUMMARY["status_counts"])
print("tasks with a verified candidate:", SUMMARY["tasks_with_candidate"])


In [ ]:
import json

from src.curriculum.submission.challenges import load_challenges
from src.curriculum.submission.format import validate_submission

_submission = json.loads(OUTPUT.read_text(encoding="utf-8"))
validate_submission(_submission, load_challenges(CHALLENGES))
print("submission.json re-read and validated:", len(_submission), "tasks,", OUTPUT.stat().st_size, "bytes")
